# 00b — Benchmark Dataset (2013–2022): Provenance and Verification

`singapore_dengue_weather_2013_2022.csv` is the benchmark dataset behind every result in
Chapter 5. It was assembled early in the project and **the original ingestion script was not
retained**. This notebook exists to close that gap. It does not reconstruct the file; it
verifies it, and it makes the reproducible half of it genuinely reproducible.

**What can be re-derived from source**

| Part | Status |
|---|---|
| Meteorological columns (temperature, rainfall, humidity, wind) | **Re-fetchable** from the Open-Meteo ERA5 archive — Section 3 does this and compares |
| Derived columns (7-day rolling features, calendar flags) | **Recomputable** from the raw columns — Section 5 |
| Calendar columns (`day_of_year`, `month`, `year`, `is_rainy_season`) | **Recomputable** — Section 6 |
| `population_density` | Structure auditable — Section 6 |
| `dengue_cases` | **Not re-derivable.** The 2013–2022 NEA weekly bulletins were not archived. Section 4 verifies the column's structure and reports annual totals for external checking. |

The 2023–2025 holdout has a full construction notebook
(`Singapore_Dengue_Dataset_FIXED_v7.ipynb`); this file is the 2013–2022 counterpart, limited
to what the surviving materials allow.

Run this notebook from the repository root.

## 1. Setup

In [1]:
import pandas as pd, numpy as np, json, urllib.request, warnings
warnings.filterwarnings('ignore')

CSV   = 'singapore_dengue_weather_2013_2022.csv'
LAT, LON = 1.3521, 103.8198          # Singapore
START, END = '2013-01-01', '2022-12-31'

df = pd.read_csv(CSV, parse_dates=['date']).sort_values('date').reset_index(drop=True)
print(f"{CSV}: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df.date.min().date()} to {df.date.max().date()}")
print("\nColumns:")
for c in df.columns:
    print(f"   {c}")

singapore_dengue_weather_2013_2022.csv: 3,652 rows x 18 columns
Date range: 2013-01-01 to 2022-12-31

Columns:
   date
   dengue_cases
   temperature
   temperature_max
   temperature_min
   rainfall
   humidity
   windspeed
   windspeed_max
   day_of_year
   month
   year
   population_density
   is_rainy_season
   temperature_7d_avg
   rainfall_7d_sum
   humidity_7d_avg
   cases_7d_avg


## 2. Structural integrity

In [2]:
checks = {}
expected_days = (df.date.max() - df.date.min()).days + 1
checks['row count matches calendar span'] = (len(df) == expected_days, f"{len(df):,} rows, {expected_days:,} days")
checks['no duplicate dates']             = (df.date.duplicated().sum() == 0, f"{df.date.duplicated().sum()} duplicates")
checks['no calendar gaps']               = ((df.date.diff().dt.days.dropna() != 1).sum() == 0,
                                            f"{(df.date.diff().dt.days.dropna() != 1).sum()} gaps")
checks['no missing values']              = (df.isna().sum().sum() == 0, f"{df.isna().sum().sum()} nulls")
checks['leap days present (2016, 2020)'] = (
    all(((df.date.dt.month==2) & (df.date.dt.day==29) & (df.date.dt.year==y)).any() for y in (2016, 2020)), '2016-02-29, 2020-02-29')

for name, (ok, detail) in checks.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name:34} {detail}")

  PASS  row count matches calendar span    3,652 rows, 3,652 days
  PASS  no duplicate dates                 0 duplicates
  PASS  no calendar gaps                   0 gaps
  PASS  no missing values                  0 nulls
  PASS  leap days present (2016, 2020)     2016-02-29, 2020-02-29


## 3. Meteorological columns — re-fetch from source and compare

Open-Meteo serves ERA5 reanalysis for a fixed grid cell and requires no API key, so the
weather half of this dataset is fully reproducible from source. This cell re-fetches
2013–2022 and compares against the committed CSV.

*If the machine running this notebook has no outbound network access, the cell reports that
and skips the comparison — the rest of the notebook is unaffected.*

In [3]:
def fetch_openmeteo(lat, lon, start, end):
    url = ("https://archive-api.open-meteo.com/v1/archive"
           f"?latitude={lat}&longitude={lon}&start_date={start}&end_date={end}"
           "&daily=temperature_2m_mean,temperature_2m_max,temperature_2m_min,"
           "precipitation_sum,relative_humidity_2m_mean,"
           "windspeed_10m_mean,windspeed_10m_max"
           "&timezone=Asia%2FSingapore")
    with urllib.request.urlopen(url, timeout=90) as r:
        d = json.load(r)['daily']
    return pd.DataFrame({
        'date':            pd.to_datetime(d['time']),
        'temperature':     d['temperature_2m_mean'],
        'temperature_max': d['temperature_2m_max'],
        'temperature_min': d['temperature_2m_min'],
        'rainfall':        d['precipitation_sum'],
        'humidity':        d['relative_humidity_2m_mean'],
        'windspeed':       d['windspeed_10m_mean'],
        'windspeed_max':   d['windspeed_10m_max'],
    })

try:
    api = fetch_openmeteo(LAT, LON, START, END)
    print(f"Fetched {len(api):,} days from Open-Meteo ERA5\n")
    m = df.merge(api, on='date', suffixes=('_csv', '_api'))
    print(f"{'column':18}{'max |diff|':>12}{'mean |diff|':>13}   corr")
    print('-' * 58)
    for c in ['temperature','temperature_max','temperature_min',
              'rainfall','humidity','windspeed','windspeed_max']:
        a, b = m[f'{c}_csv'], m[f'{c}_api']
        print(f"{c:18}{(a-b).abs().max():>12.3f}{(a-b).abs().mean():>13.3f}   {a.corr(b):.4f}")
    print("\nA close match confirms the meteorological columns came from this source.")
    print("Small deviations are expected: ERA5 is periodically re-analysed, and the")
    print("original download predates the current archive revision.")
except Exception as e:
    print(f"Open-Meteo not reachable from this environment ({type(e).__name__}).")
    print("Run this cell on a machine with internet access to complete the comparison.")
    print("Everything below runs offline and is unaffected.")

Fetched 3,652 days from Open-Meteo ERA5

column              max |diff|  mean |diff|   corr
----------------------------------------------------------
temperature              1.062        0.001   0.9995
temperature_max          3.300        0.018   0.9979
temperature_min          0.925        0.016   0.9990
rainfall                68.025        0.042   0.9912
humidity                 4.500        0.006   0.9995
windspeed                4.287        0.004   0.9993
windspeed_max            8.800        0.007   0.9987

A close match confirms the meteorological columns came from this source.
Small deviations are expected: ERA5 is periodically re-analysed, and the
original download predates the current archive revision.


## 4. `dengue_cases` — structure and totals

The NEA reports dengue weekly. In this dataset the weekly figure is spread evenly across
the seven days of its reporting week, so the daily series should be a step function with a
run length of 7.

In [4]:
v = df.dengue_cases.values
runs, c = [], 1
for i in range(1, len(v)):
    if v[i] == v[i-1]: c += 1
    else: runs.append(c); c = 1
runs.append(c)
runs = np.array(runs)

print("Weekly step structure")
print(f"   constant-value runs : {len(runs)}")
print(f"   mean run length     : {runs.mean():.2f}  (expected ~7)")
print(f"   median run length   : {np.median(runs):.0f}")
print(f"   runs of exactly 7   : {(runs == 7).sum()} of {len(runs)} ({(runs==7).mean()*100:.1f}%)")
print(f"   implied weeks       : {len(runs)}  (10 years ~ 522 weeks)")

print("\nAnnual totals (for cross-checking against the MOH Weekly Infectious Diseases Bulletin)")
yr = df.groupby(df.date.dt.year)['dengue_cases'].sum()
print(f"\n   {'year':6}{'dataset total':>16}")
for y, s in yr.items():
    print(f"   {y:<6}{s:>16,.0f}")
print(f"   {'':6}{'':>16}")
print(f"   {'total':6}{yr.sum():>16,.0f}")

Weekly step structure
   constant-value runs : 523
   mean run length     : 6.98  (expected ~7)
   median run length   : 7
   runs of exactly 7   : 506 of 523 (96.7%)
   implied weeks       : 523  (10 years ~ 522 weeks)

Annual totals (for cross-checking against the MOH Weekly Infectious Diseases Bulletin)

   year     dataset total
   2013            22,193
   2014            18,345
   2015            11,289
   2016            13,276
   2017             2,788
   2018             3,332
   2019            16,069
   2020            35,129
   2021             5,308
   2022            32,186
                         
   total          159,916


> **To complete this check:** fill in the published annual figures from the MOH *Weekly
> Infectious Diseases Bulletin* (or the NEA dengue statistics page) and compare. Section
> 4.2.1 of the thesis reports the same check for the 2023–2025 holdout, where the dataset
> totals fall within 0.5% of the published figures. The deviation arises from the
> weekly-to-daily conversion at year boundaries and is expected.

## 5. Derived columns — recompute from the raw columns

If these reproduce exactly, the derived features were generated from the raw columns in
this same file and contain no external input.

In [5]:
recipes = [
    ('temperature_7d_avg', 'temperature',   lambda s: s.rolling(7, min_periods=1).mean()),
    ('rainfall_7d_sum',    'rainfall',      lambda s: s.rolling(7, min_periods=1).sum()),
    ('humidity_7d_avg',    'humidity',      lambda s: s.rolling(7, min_periods=1).mean()),
    ('cases_7d_avg',       'dengue_cases',  lambda s: s.rolling(7, min_periods=1).mean()),
]
print(f"{'derived column':22}{'source':16}{'max |diff|':>13}   result")
print('-' * 66)
for out, src, fn in recipes:
    d = (fn(df[src]) - df[out]).abs().max()
    print(f"{out:22}{src:16}{d:>13.2e}   {'exact match' if d < 1e-9 else 'MISMATCH'}")

derived column        source             max |diff|   result
------------------------------------------------------------------
temperature_7d_avg    temperature          1.07e-14   exact match
rainfall_7d_sum       rainfall             2.84e-14   exact match
humidity_7d_avg       humidity             1.42e-14   exact match
cases_7d_avg          dengue_cases         5.68e-14   exact match


## 6. Calendar and population columns

In [6]:
print("Calendar columns")
for c, expected in [('day_of_year', df.date.dt.dayofyear),
                    ('month',       df.date.dt.month),
                    ('year',        df.date.dt.year)]:
    ok = (df[c].astype(int) == expected).all()
    print(f"   {'PASS' if ok else 'FAIL'}  {c}")

months = sorted(int(m) for m in df.loc[df.is_rainy_season == 1, 'date'].dt.month.unique())
allmatch = (df.is_rainy_season.astype(int) == df.date.dt.month.isin(months).astype(int)).all()
print(f"   {'PASS' if allmatch else 'FAIL'}  is_rainy_season = months {months} "
      f"({df.is_rainy_season.mean()*100:.1f}% of days)")
print("        -> the Northeast Monsoon window, applied as a pure calendar rule")

print("\npopulation_density")
g = df.groupby(df.date.dt.year)['population_density'].agg(['first', 'nunique'])
for y, row in g.iterrows():
    print(f"   {y}   {row['first']:>10.2f}   distinct values within year: {int(row['nunique'])}")
print(f"\n   distinct values across the full period: {df.population_density.nunique()}")

Calendar columns
   PASS  day_of_year
   PASS  month
   PASS  year
   PASS  is_rainy_season = months [1, 2, 3, 11, 12] (41.4% of days)
        -> the Northeast Monsoon window, applied as a pure calendar rule

population_density
   2013      7364.89   distinct values within year: 1
   2014      7461.06   distinct values within year: 1
   2015      7550.13   distinct values within year: 1
   2016      7648.75   distinct values within year: 1
   2017      7655.57   distinct values within year: 1
   2018      7691.58   distinct values within year: 1
   2019      7780.11   distinct values within year: 1
   2020      7755.83   distinct values within year: 1
   2021      7439.09   distinct values within year: 1
   2022      7689.26   distinct values within year: 1

   distinct values across the full period: 10


> **Note for Section 4.2.1.** The dataset holds **one constant value per calendar year**
> (10 distinct values over 3,652 days), i.e. an annual step series. Section 4.2.1 currently
> describes this column as *"interpolated to daily"*, which would produce a smoothly varying
> value on every date. The wording should be corrected to reflect what the data does — the
> SingStat annual figure held constant across each year. The column is a weak, near-constant
> predictor either way, so no result changes.

## 7. Summary

In [7]:
print("Benchmark dataset 2013-2022 - verification summary")
print("=" * 58)
print("  Structure          3,652 daily rows, no gaps, no duplicates, no nulls")
print("  Weather columns    re-fetchable from Open-Meteo ERA5 (Section 3)")
print("  Derived columns    reproduce exactly from raw columns (Section 5)")
print("  Calendar columns   reproduce exactly from the date index (Section 6)")
print("  population_density annual step series, 10 distinct values (Section 6)")
print("  dengue_cases       weekly step structure confirmed; source bulletins")
print("                     for 2013-2022 were not archived, so the column")
print("                     cannot be re-derived from source")
print()
print("  Limitation: this notebook verifies the benchmark dataset. It does not")
print("  reconstruct it. The construction script was not retained.")

Benchmark dataset 2013-2022 - verification summary
  Structure          3,652 daily rows, no gaps, no duplicates, no nulls
  Weather columns    re-fetchable from Open-Meteo ERA5 (Section 3)
  Derived columns    reproduce exactly from raw columns (Section 5)
  Calendar columns   reproduce exactly from the date index (Section 6)
  population_density annual step series, 10 distinct values (Section 6)
  dengue_cases       weekly step structure confirmed; source bulletins
                     for 2013-2022 were not archived, so the column
                     cannot be re-derived from source

  Limitation: this notebook verifies the benchmark dataset. It does not
  reconstruct it. The construction script was not retained.
